<h1 style="text-align:center;">TREĆI PROJEKAT</h1>

<h2 style="font-style:italic; font-weight:bold; text-align: center;">
    Analiza sentimenata recenzija filmova sa platforme IMDB primenom neuronskih mreža
</h2>

## 1. Uvod
Cilj ovog projekta je primena veštačkih neuronskih mreža za analizu sentimenata tekstualnih podataka. Analiza sentimenta predstavlja zadatak obrade prirodnog jezika (NLP) čiji je cilj klasifikacija tekstualnih podataka na osnovu izraženog sentimenta. U ovom slučaju potrebno je utvrditi da li filmska recenzija ima pozitivan ili negativan sentiment.

Za potrebe ovog rada korišćene je IMDB dataset kojib sadrži 50000 filmskih recenzija. Svaka recenzija je označena kao pozitivna ili negativna.

Dataset je preuzet sa Kaggle platforme: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

Za implementaciju modela korišćene su sledeće biblioteke:
- Python
- Pandas
- Scikit-learn
- TensorFlow / Keras
- Matplotlib i Seaborn

U nastavku rada prikazana je analiza dataset-a, priprema podataka, implementacija neuronske mreže, treniranje modela i evaluacija dobijenih rezultata.

## 2. Opis skupa podataka
Dataset koji je korišćen u ovom projektu sadrži 50 000 filmskih recenzija preuzetih sa platforme IMDB.. Svaka recenzija je označena kao pozitivna ili negativna u zavisnosti od sentimenta koji izražava.

Dataset sadrži dve kolone:
- **review** - tesktualni sadržaj filmske recenzije
- **sentiment** - oznaka sentimenta (positive ili negative)

Podaci su uravnoteženi, što znači da postoji približno jedank broj pozitivnih i negativnih recenzija. To je važno jer omogućava modelu da uči bez pristrasnosti prema jednoj klasi.

Pre treniranja modela potrebno je izvršiti osnovnu analizu i pripremu podataka kako bi se tesktualni podaci mogli koristiti u modelu neuronske mreže.

## 3. Učitavanje biblioteka

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from wordcloud import STOPWORDS
from collections import Counter
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, Input
from tensorflow.keras.optimizers import Adam

## 4. Učitavanje skupa podataka
U ovom koraku učitavamo dataset u pandas DataFrame kako bismo mogli da analiziramo i obradimo podatke.

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Prikaz prvih nekoliko redova dataset-a omogućava uvid u strukturu podataka i tipove vrednosti koje dataset sadrži.

## 5. Deskriptivna analiza
Pre treniranja modela potrebno je analizirati osnovne karakteristike dataset-a kao što su broj instanci, distribucija klasa i eventualne nedostajuće vrednosti.

### Dimenzije dataset-a

In [3]:
df.shape

(50000, 2)

Dataset sadrži 50000 instanci i 2 kolone: tekst recenzije i oznaki sentimenta.

### Provera nedostajućih vrednosti

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

Rezultati pokazuju da dataset ne sadrži nedostajuče vrenosti, što znači da nije potrebna dodatna obrada podataka u ovom koraku.

## 6. Primeru recenzija iz dataset-a

U ovom delu prikazujemo nekoliko nasumičnih recenzija iz dataset-a kako bismo dobili bolji uvid u tekstualne podatke koji se koriste za treniranje modela.

In [5]:
print("Random positive review:")
print(df[df['sentiment']==1]['review'].sample(1).values[0])

print("\nRandom negative review:")
print(df[df['sentiment']==0]['review'].sample(1).values[0])

Random positive review:


ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
print("Random positive reviews:")
print(df[df['sentiment']==1]['review'].sample(3).values)

print("\nRandom negative reviews:")
print(df[df['sentiment']==0]['review'].sample(3).values)

## 7.  Distribucija sentimenta 

In [ ]:
df['sentiment'].value_counts()

### Vizuelizacija

In [ ]:
sns.countplot(x='sentiment', data = df)
plt.title("Distribucija sentimenta u dataset-u")
plt.xlabel("Sentiment")
plt.ylabel("Broj recenzija")
plt.show()

Iz grafika se može videti da dataset sadrži približno jednak broj pozitivnih i negativnih recenzija, što znači da je dataset uravnotežen. Ovakva struktura podataka je pogodna za treniranje klasifikacionog modela.

## 8. WordCloud analiza recenzija
WordCloud predstavlja vizuelizacija najčešće korišćenih reči u tekstualnim podacima. Veće reči u grafiku označavaju reči koji se češće pojavljuju u recenzijama.

U nastavku su prikazani WordCloud grafici za pozitivne i negativne filmske recenzije.

### WordCloud za pozitivne recenzije

In [ ]:
positive_reviews = df[df['sentiment'] == 'positive']['review']
text = " ".join(positive_reviews)
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    stopwords=STOPWORDS
).generate(text)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("WordCloud - Positive Reviews")
plt.show()

### WorldCloud za negativne recenzije

In [ ]:
negative_reviews = df[df['sentiment'] == 'negative']['review']
text = " ".join(negative_reviews)
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='black',
    stopwords=STOPWORDS
).generate(text)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("WordCloud - Negative Reviews")
plt.show()

Iz WordCloud grafika može se uočiti koje reči se najčešće pojavljuju u pozitivnim i negativnim recenzijama.

U pozitivnim recenzijama dominiraju reči koje ukazuju na zadovoljstvo gledalaca, dok se u negativnim recenzijama češće pojavljuju reči koje ukazuju na kristiku filma ili loše iskustvo gledanja.

## 9. Analiza dužine recenzija
U ovom delu analiziramo dužinu recenzija u dataset-u. Dužina recenzije predstavlja broj reči u svakoj recenziji. Ova analiza može pomoći u razumevanju strukture tektusalnih podataka.

In [ ]:
df['review_length'] = df['review'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,5))
sns.histplot(df['review_length'], bins=50)

plt.title("Distribucija dužine recenzija")
plt.xlabel("Broj reči")
plt.ylabel("Broj recenzija")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='sentiment', y='review_length', data=df)

plt.title("Dužina recenzija po sentimentu")

plt.show()

Iz grafika se može videti raspodela dužine u dataset-u. Većina recenzija sadrži između odreženog broja reči, dok postoje i duže recenzije. Analiza pokazuje da dataset sadrži raznovrsne tekstualne primere, što je korisno za treniranje modela.

## 10. Najčešće reči u recenzijama
U ovom delu analiziraju se najčesšće reči koje se pojavljuju u filmskim recenzijama. Ova analiza daje dodatni uvid u strukturz tesktualnih podataka.

In [ ]:
text = " ".join(df['review']).lower()
words = re.findall(r'\w+', text)
word_counts = Counter(words)
common_words = word_counts.most_common(20)
words = [word[0] for word in common_words]
counts = [word[1] for word in common_words]

plt.figure(figsize=(10,5))
plt.bar(words, counts)
plt.title("Top 20 najčešćih reči u recenzijama")
plt.xlabel("Reči")
plt.ylabel("Frekvencija pojavljivanja")
plt.xticks(rotation=45)
plt.show()

Iz grafa se moože primetiti da se među najčešćim rečima nalaze opšte reči enegleskog jezika kao što su "the", "and", "a", "of". Ove reči nemaju veliku semantičku vrednost za analizu sentimenta, zbog čega se u mnogim NLP motodama koriste tehnike uklanjanja stop reči pre treniranja modela.

## 11. Najčešće reči u pozitivnim i negativnim recenzijama

In [ ]:
positive_reviews = " ".join(df[df['sentiment'] == 1]['review']).lower()
negative_reviews = " ".join(df[df['sentiment'] == 0]['review']).lower()
pos_words = re.findall(r'\w+', positive_reviews)
neg_words = re.findall(r'\w+', negative_reviews)
pos_counts = Counter(pos_words).most_common(15)
neg_counts = Counter(neg_words).most_common(15)

print("Top positive words:")
print(pos_counts)
print("\nTop negative words:")
print(neg_counts)

In [ ]:
#pozitivne
pos_words = [word[0] for word in pos_counts]
pos_freq = [word[1] for word in pos_counts]

plt.figure(figsize=(10,5))
plt.bar(pos_words, pos_freq)
plt.title("Najčešće reči u pozitivnim recenzijama")
plt.xticks(rotation=45)
plt.show()

In [ ]:
#negativne
neg_words = [word[0] for word in neg_counts]
neg_freq = [word[1] for word in neg_counts]

plt.figure(figsize=(10,5))
plt.bar(neg_words, neg_freq)
plt.title("Najčešće reči u negativnim recenzijama")
plt.xticks(rotation=45)
plt.show()

## 12. Priprema podataka
Tekstualni podaci moraju biti pretvoreni u numerički oblik kako bi neuronska mreža mogal da ih obradi.

Koraci pripreme podataka:
- Pretvaranje sentimenata u numeričke vrednosti
- Tokenizaciju teksta
- Pretvaranje teksta u sekvence brojeva
- Padding sekvenci

### Kodiranje sentimenata

In [ ]:
encoder = LabelEncoder()
df['sentiment'] = encoder.fit_transform(df['sentiment'])
df.head()

Sentiment oznake su pretvorene u numeričke vrednosti kako bi ih neuronska ,reža mogla koristiti u procesu treniranja. 
- Positive → 1
- Negative → 0

### Tokenizacija teksta
Tokenizacija predstavlja proces pretvranja teksta u niz tokena (reči) koji se zatim mapiraju na numeričke vrednosti.

In [ ]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df['review'])
X = tokenizer.texts_to_sequences(df['review'])

### Padding sekvenci

In [ ]:
X = pad_sequences(X, maxlen = 200)

Padding se koristi kako bi se sekvence imale istu dužinu. U ovom slučaju maksimalna dužina sekvence je postavljena na 200, što omogućava modelu da efikasno obrađuje tekstulane podatke.

In [ ]:
X.shape

In [ ]:
#target promenljiva
y = df['sentiment']

### Podela na trening i test skup

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Podaci su podeljeni na trening i test skup:
- Trening skup se koristi za treniranje neuronske mreže
- Test skup se koristi za eveluaciju performansi modela.

U ovom projektu korišćen je odnos 80% trenig podataka i 20% test podataka.

### 13. Kreiranje neuronske mreže
Za analizu sentimenata koristićemo LSTM (Long Short - Term Memory) neuronsku mrežu. 

LSTM mreže su posebno pogodne za obradu sekvencijalnih podataka kao što su tekstualni podaci jer mogu da pamte kontekst reči u rečenici.

In [ ]:
model = Sequential()
model.add(Input(shape=(200,)))
model.add(Embedding(input_dim=10000, output_dim=128))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

Model neuronske mreže sastoji se od nekoliko slojeva:
- Embedding sloj pretvara reči u numeričke vektore
- LSTM sloj omgućava modelu da pamti kontekst reči u sekvenci
- Dropout sloj se korsiti za smanjenje overfitting-a tokom treniranja.
- DEnse sloj ssa sigmoid aktivacijom daje verovatnoću  da recenzija imapozitivan sentiment.

## 14. Kompajliranje modela
Pre treniranja modela potrebno je definisati:
- optimizator
- loss funkciju
- metriku evaluacije

Za ovaj problem koristi se **binary_crossentropy** jer se radi o binarnoj klasifikaciji.

In [ ]:
model.compile(
    loss = 'binary_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)
model.summary()

Iz prikaza arhitekture modela može se videti broj slojeva, dimenzije izlaza svakog sloja i broj parametara koji se treniraju.

Ukupan broj parametara modela iznosi 1,329,473 što predstavlja parametre koje neuronska mreža uči tokom procesa treniranja.

## 15. Treniranje modela

Model se trenira korišćenjem trening skupa podataka. Tokom treniranja model prolazi kroz više epoha (epochs) kako bi postepeno učio obrasce u tekstualnim podacima.

Parametar bath_size određuje koliko uzoraka model obrađuje u jednom koraku treniranja.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs = 5,
    batch_size = 64,
    validation_split = 0.2
)

Tokom treniranja može se primetiti da se tačnost modela postepeno povećava dok se vrednost loss funkcije smanjuje.

Validation accuracy pokazuje koliko model dobro generalizuje na podacima koji nisu korišćeni tokom treniranja.

## 16. Vuzuelizacija rezultata
### Accuracy graf

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])

plt.show()

Iz grafa tačnosti može se videti tačnost modela postepeno povećava tokom treniranja.

Razlika između trening i validacionog skupa pokazuje koliko model uspešno generalizuje na novim podacima.

### Loss graf

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])

plt.show()

Graf loss funkcije pokazuje kako se greška modela menja tokom treniranja .

Smanjenje vrednosti loss funkcije na trening skupu ukazuje na to da model uspešno uči obrasce u podacima, dok vrednosti na validacionom skupu pokazuje sposobnost modela da generalizuje na neviđene podatke.

## 17. Evaluacija modela 

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test accuracy:", accuracy)

Rezultati evaluacije pokazuje da model postiže tačnost od približno 87.6% na test skup podataka.

Ovaj rezultat ukazuje na model uspešno klasifikuje većinu pozitivnih i negativnih filmskih recenzija koje nisu korišćene tokom treniranja.

## 18. Confusion matrix
Confusion matrix predstavja tabelu koja prikazuje performanse klasifikacionog modela poređenjem stvarnih i predviđenih vrednosti.

Matrica omogućava bolju analizu tačnosti modela i prikazuje:
- True positive (TP)
- True negative (TN)
- False positive (FP)
- False negative (FN)

In [ ]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

Iz confusion matrix može se videti raspodela tačno i pogrešno klasifikovanih recenzija. Model je uspešno klasifikovao veliki broj pozitivnih i negativnih recenzija, dok manji broj recenzija nije pravilno klasifikovan.

Ova analiza omogućava bolji uvid u performase modela u odnosu na pojedinačne klase.

In [ ]:
print(classification_report(y_test, y_pred))

Classification report prikazuje detaljne metrike performansi modela:
- Precision - odnos tačno predviđenih pozitivnih instanci i svih predviđenih pozitivnih instanci
- Recall - odnos tačno predviđenih pozitivnih instanci i svih stvarnih pozitivnih instanci
- F1-score - harmonijska sredina precision i recall metrike

Rezultati pokazuju da model postiže veoma dobre performanse u klasifikaciji sentimenata filmskih recenzija.

## 19. ROC AUC analiza
ROC kriva prikazuje odnos između True Positive Rate i False Positive Rate za različite pragove klasifikacije. AUC vrednost predstavlja meru kvaliteta modela.

In [ ]:
y_pred_prob = model.predict(X_test)
fpr, tpr, threshold = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label="ROC curve (AUC = %0.2f)" % roc_auc)
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

Iz ROC krive može se videti odnos između True Positive Rate i False Positive Rate za različite pragoove klasifikacije.

Vrednost AUC iznosi približno 0.94, što ukazuje na veoma dobre performanse modela u razlikovanju pozitivnih i negativnih recenzija.

Što je AUC vrednost bliža 1, to model ima bolju sposobnost klasifikacije.

## 20. Sumirani rezultati modela

In [ ]:
acuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

## 21. Predikcija sentimenata nove recenzije
Model se može koristiti i za predikciju sentimenata novih filmskih recenzija koje nisu deo trening skupa.

In [ ]:
sample_review = ["This movie was amazing and I really enjoyed it"]
sequence = tokenizer.texts_to_sequences(sample_review)
padded = pad_sequences(sequence, maxlen=200)
prediction = model.predict(padded)
if prediction > 0.5:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

U ovom primeru model je korišćen za predikciju sentimenata nove filmske recenzije koja nije deo trening skupa. Tekst recenzije se prvo pretvara u numeričku sekvencu koristeći toknizer, zatim se sekvenca prilagođava na odgovarajuču dužinu korišćenjem padding tehnike.

Na osnovu dobijene verovatnoće model određuje da li recenzija ima pozitavna ili negativan sentiment.

In [ ]:
sample_review = ["This movie was terrible and boring"]
sequence = tokenizer.texts_to_sequences(sample_review)
padded = pad_sequences(sequence, maxlen=200)
prediction = model.predict(padded)
if prediction > 0.5:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

## 22. Analiza rezultata
Rezultati pokazuju da neuronska mreža uspešno klasifikuje sentiment filmskih recenzija. Model postiže visoku tačnost na test skup podataka, što pokazuje da LSTM mreže mogu efikasno da prepoznaju sentiment u tekstualnim podacima.

Vizuelizacija accuracy i loss funkcije pokazuje stabilno učenje modela, uz blagu razliku između trening i validacionih rezultata.

## 23. Zaključak
U ovom projektu implementirana je veštačka neuronska mreža za analizu sentimenta filmskih recenzija. Korišećenjem LSTM neurosnke mreže moguće je uspešno analizirati tekstualne podatke i klasifikovati sentiment recenzija kao pozitivan ili negativan.

Rezultati pokazuju da neuronske mreže predstavljaju veoma efikasan pristup za obradu prirodnog jezika i analizu sentimenta.